# Data Cleaning R&D

Goal:
Test common cleaning steps across multiple insurance documents before chunking.

We will avoid company-specific cleaning and focus on generic rules.

In [1]:
import re
import os
import pandas as pd
from langchain_community.document_loaders import PyMuPDFLoader

C:\Users\Vikram\AppData\Local\Temp\ipykernel_33644\645806584.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
c:\Users\Vikram\insurance_rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sample_files = [
    "../data/general/health/SBI_Health_Insurance_Retail.pdf",
    "../data/life/LIC_Tech_Term_Plan.pdf",
    "../data/general/motor/HDFC_ERGO_Motor_Insurance.pdf",
    "../data/general/travel/Bajaj_Allianz_Travel_Insurance.pdf",
]

In [3]:
def load_pdf_text(file_path):
    loader = PyMuPDFLoader(file_path)
    docs = loader.load()
    text = "\n".join([doc.page_content for doc in docs])
    return text

raw_outputs = {}

for file in sample_files:
    if os.path.exists(file):
        raw_outputs[file] = load_pdf_text(file)
        print("Loaded:", file, "| Length:", len(raw_outputs[file]))
    else:
        print("Missing:", file)

Loaded: ../data/general/health/SBI_Health_Insurance_Retail.pdf | Length: 92847
Loaded: ../data/life/LIC_Tech_Term_Plan.pdf | Length: 3207
Loaded: ../data/general/motor/HDFC_ERGO_Motor_Insurance.pdf | Length: 2711
Loaded: ../data/general/travel/Bajaj_Allianz_Travel_Insurance.pdf | Length: 2110


In [4]:
for file, text in raw_outputs.items():
    print("\n" + "="*100)
    print("FILE:", file)
    print("="*100)
    print(text[:1500])


FILE: ../data/general/health/SBI_Health_Insurance_Retail.pdf
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company Limited
Limited
Limited
Limited
                 
 
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited  
SBI General Insurance Company Limited   
Corporate & Registered Office:   'Natraj', 301, Junction of  Western Express Highway & Andheri - Kurla Road, Andheri (E), Mumbai - 400 069 I 
CIN: U66000MH2009PLC190546 I    Tel.: +91 22 42412000 I  
 www.sbigeneral.in I Logo displayed belongs to State Bank of India and is used 
by SBI General Insurance Co. Ltd. under license I IRDAI Registration Number 144 I  Product Name: Health Insurance Policy Retail. I  
UIN: SBIHLIP11002V021011 I IRDAI Reg No.144 
 
HEALTH INSURANCE POLICY –RETAIL 
 
This Policy is issued to the Insured based on the Proposal and declaration together with any statement, rep

In [7]:
import re

def clean_text(text):

    # Replace weird bullet symbols
    text = text.replace("\uf0b7", "•")

    # Remove repeated blank lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # Remove excessive spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces before punctuation
    text = re.sub(r"\s+([,.!?;:])", r"\1", text)

    # Remove page number patterns
    text = re.sub(
        r"Page\s+\d+\s+of\s+\d+",
        "",
        text,
        flags=re.IGNORECASE
    )

    return text.strip()

In [8]:
cleaned_outputs = {}

for file, raw_text in raw_outputs.items():
    cleaned_outputs[file] = clean_text(raw_text)

In [9]:
for file, text in cleaned_outputs.items():
    print("\n" + "="*100)
    print("FILE:", file)
    print("="*100)
    print(text[:1500])


FILE: ../data/general/health/SBI_Health_Insurance_Retail.pdf
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company 
SBI General Insurance Company Limited
Limited
Limited
Limited

SBI General Insurance Company Limited 
SBI General Insurance Company Limited 
SBI General Insurance Company Limited 
SBI General Insurance Company Limited 
Corporate & Registered Office: 'Natraj', 301, Junction of Western Express Highway & Andheri - Kurla Road, Andheri (E), Mumbai - 400 069 I 
CIN: U66000MH2009PLC190546 I Tel.: +91 22 42412000 I 
 www.sbigeneral.in I Logo displayed belongs to State Bank of India and is used 
by SBI General Insurance Co. Ltd. under license I IRDAI Registration Number 144 I Product Name: Health Insurance Policy Retail. I 
UIN: SBIHLIP11002V021011 I IRDAI Reg No.144 

HEALTH INSURANCE POLICY –RETAIL 

This Policy is issued to the Insured based on the Proposal and declaration together with any statement, report or 
other document which shall 

# Cleaning Observation

The generic cleaning function improved whitespace, blank lines, punctuation spacing, and unicode artifacts.

However, repeated company headers still remain in some PDFs.

Decision:
We will not remove company names using hardcoded rules because company names can be useful for policy context and source identification.

For production:
- Use generic cleaning by default.
- Handle repeated headers later if they appear across many documents.
- Avoid over-cleaning important policy/legal terms.

In [10]:
summary = []

for file in raw_outputs:
    raw_len = len(raw_outputs[file])
    clean_len = len(cleaned_outputs[file])

    summary.append({
        "file": os.path.basename(file),
        "raw_length": raw_len,
        "cleaned_length": clean_len,
        "characters_removed": raw_len - clean_len,
        "reduction_percent": round(((raw_len - clean_len) / raw_len) * 100, 2)
    })

pd.DataFrame(summary)

,file,raw_length,cleaned_length,characters_removed,reduction_percent
0,SBI_Health_Insurance_Retail.pdf,92847,91586,1261,1.36
1,LIC_Tech_Term_Plan.pdf,3207,3207,0,0.00
2,HDFC_ERGO_Motor_Insurance.pdf,2711,2711,0,0.00
3,Bajaj_Allianz_Travel_Insurance.pdf,2110,2110,0,0.00


# Final Cleaning R&D Conclusion

Documents Tested:
- SBI Health Insurance Retail
- LIC Tech Term Plan
- HDFC ERGO Motor Insurance
- Bajaj Allianz Travel Insurance

Findings:
- Most insurance documents were already reasonably clean after extraction using PyMuPDFLoader.
- Generic cleaning removed minor whitespace and formatting artifacts.
- SBI Health Insurance document showed the highest improvement (1.36% reduction).
- Other documents required little to no cleaning.

Decision:
Use lightweight generic cleaning.

Cleaning Rules:
- Normalize whitespace
- Remove excessive blank lines
- Fix unicode artifacts
- Remove page numbering patterns

Avoid:
- Company-specific text removal
- Aggressive cleaning
- Removing legal/policy terminology

Reason:
Over-cleaning may remove important information required for retrieval and policy recommendation.